In [1]:
# weather + accidents

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
# spark = SparkSession.builder.appName("NYC").config("spark.jars", "/jars/postgresql-42.7.3.jar").getOrCreate()
# spark = SparkSession.builder.appName("NYC").config("spark.jars.packages", "org.postgresql:postgresql:42.7.3").getOrCreate()
spark = SparkSession.builder.appName("NYC").getOrCreate()

base_path = "/home/jovyan/work"

In [4]:
spark._jvm.java.lang.Class.forName("org.postgresql.Driver")

JavaObject id=o28

In [5]:
weather_df = spark.read.parquet(f"{base_path}/CLEANED_weather_partitioned/")
collision_df = spark.read.parquet(f"{base_path}/CLEANED_vehicle_collisions_partitioned/")

weather_df_filtered = weather_df.filter((weather_df.year == 2023) | (weather_df.year == 2024) | (weather_df.year == 2025))
collision_df_filtered = collision_df.filter((collision_df.year == 2023) | (collision_df.year == 2024) | (collision_df.year == 2025))

In [6]:
weather_df_filtered.sort("Date", ascending=True) #.show(3, vertical=True)

DataFrame[REPORT_TYPE: string, DailyWeather: string, HourlyDryBulbTemperature: double, HourlyPrecipitation: string, Date: date, Time: string, daily_avg_temp: double, daily_max_temp: double, daily_min_temp: double, daily_precipitation: string, daily_sunrise: double, daily_sunset: double, year: int, month: int]

In [7]:
collision_df_filtered.sort("CRASH DATE", ascending=True) #.show(3, vertical=True)

DataFrame[CRASH DATE: date, CRASH TIME: string, BOROUGH: string, LOCATION: string, ON STREET NAME: string, CROSS STREET NAME: string, OFF STREET NAME: string, NUMBER OF PERSONS INJURED: double, NUMBER OF PERSONS KILLED: double, NUMBER OF PEDESTRIANS INJURED: bigint, NUMBER OF PEDESTRIANS KILLED: bigint, NUMBER OF CYCLIST INJURED: bigint, NUMBER OF CYCLIST KILLED: bigint, NUMBER OF MOTORIST INJURED: bigint, NUMBER OF MOTORIST KILLED: bigint, CONTRIBUTING FACTOR VEHICLE 1: string, CONTRIBUTING FACTOR VEHICLE 2: string, CONTRIBUTING FACTOR VEHICLE 3: string, CONTRIBUTING FACTOR VEHICLE 4: string, CONTRIBUTING FACTOR VEHICLE 5: string, COLLISION_ID: bigint, VEHICLE TYPE CODE 1: string, VEHICLE TYPE CODE 2: string, VEHICLE TYPE CODE 3: string, VEHICLE TYPE CODE 4: string, VEHICLE TYPE CODE 5: string, year: int, month: int]

In [8]:
weather_hourly = weather_df_filtered.withColumn(
    "hourly_precip_num",
    when(col("HourlyPrecipitation") == "T", 0.001)
    .otherwise(col("HourlyPrecipitation").cast("double"))
)

weather_hourly = weather_hourly.withColumn(
    "weather_condition",
    when(col("hourly_precip_num") > 0, "Rain")
    .otherwise("No Rain")
)

In [9]:
# Cuts "11:00" at ":" -> Keeps 11
collision_df_filtered = collision_df_filtered.withColumn(
    "crash_hour", 
    split(col("CRASH TIME"), ":")[0].cast("int")
)

# Cuts "23:51:00" at ":"-> Keeps 23
weather_hourly = weather_hourly.withColumn(
    "weather_hour", 
    split(col("Time"), ":")[0].cast("int")
)

joined_df = collision_df_filtered.join(
    weather_hourly.select("Date", "weather_hour", "weather_condition"),
    (collision_df_filtered["CRASH DATE"] == weather_hourly["Date"]) & (collision_df_filtered["crash_hour"] == weather_hourly["weather_hour"]),
    "inner"
)

In [10]:
weather_daily = weather_hourly.groupBy("Date").agg(
    max(
        when(col("hourly_precip_num") > 0, 1)
        .otherwise(0)
    ).alias("rain_day")
)

daily_collisions = collision_df_filtered.groupBy(
    col("CRASH DATE").alias("Date")
).agg(
    count("*").alias("daily_collisions")
)

daily_joined = daily_collisions.join(
    weather_daily,
    "Date",
    "inner"
)

In [11]:
daily_result = daily_joined.groupBy("rain_day").agg(
    avg("daily_collisions").alias("avg_collisions_per_day"),
    count("*").alias("num_days")
)

daily_result.show()

+--------+----------------------+--------+
|rain_day|avg_collisions_per_day|num_days|
+--------+----------------------+--------+
|       1|    236.23347107438016|     484|
|       0|    234.99016393442622|     610|
+--------+----------------------+--------+



In [12]:
hour_counts = weather_hourly.groupBy(
    "weather_condition"
).agg(
    count("*").alias("total_hours")
)

collision_counts = joined_df.groupBy(
    "weather_condition"
).agg(
    count("*").alias("total_collisions")
)


In [13]:
#collisions in rainy vs dry hours
hourly_result = collision_counts.join(
    hour_counts,
    "weather_condition"
).withColumn(
    "collisions_per_hour",
    col("total_collisions") / col("total_hours")
)

hourly_result.show()

+-----------------+----------------+-----------+-------------------+
|weather_condition|total_collisions|total_hours|collisions_per_hour|
+-----------------+----------------+-----------+-------------------+
|          No Rain|          256030|      26618|  9.618679089338041|
|             Rain|           76099|       7098| 10.721189067342914|
+-----------------+----------------+-----------+-------------------+



In [14]:
hourly_result.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/traffic") \
    .option("dbtable", "hourly_weather_accidents") \
    .option("user", "admin") \
    .option("password", "admin") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

In [15]:
df_check = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/traffic") \
    .option("dbtable", "hourly_weather_accidents") \
    .option("user", "admin") \
    .option("password", "admin") \
    .option("driver", "org.postgresql.Driver") \
    .load()

df_check.show()

+-----------------+----------------+-----------+-------------------+
|weather_condition|total_collisions|total_hours|collisions_per_hour|
+-----------------+----------------+-----------+-------------------+
|          No Rain|          256030|      26618|  9.618679089338041|
|             Rain|           76099|       7098| 10.721189067342914|
+-----------------+----------------+-----------+-------------------+

